In [1]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, f1_score
from imblearn.over_sampling import SMOTE
import joblib

from src.model_utils import evaluate_model


ModuleNotFoundError: No module named 'src'

In [ ]:
#  Load cleaned datasets (after preprocessing from Task 1)
fraud_df = pd.read_csv("../../data/processed/fraud_data_cleaned.csv")
credit_df = pd.read_csv("../../data/processed/creditcard_cleaned.csv")


In [ ]:
#  Separate features and target
X_fraud = fraud_df.drop(columns=["class"])
y_fraud = fraud_df["class"]

X_credit = credit_df.drop(columns=["Class"])
y_credit = credit_df["Class"]


In [ ]:
# Train-test split
Xf_train, Xf_test, yf_train, yf_test = train_test_split(X_fraud, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_credit, y_credit, test_size=0.2, random_state=42, stratify=y_credit)


In [ ]:
#  Apply SMOTE to handle imbalance
smote = SMOTE(random_state=42)
Xf_train_res, yf_train_res = smote.fit_resample(Xf_train, yf_train)
Xc_train_res, yc_train_res = smote.fit_resample(Xc_train, yc_train)


In [ ]:
#  Scaling (important for Logistic Regression)
scaler_fraud = StandardScaler()
Xf_train_scaled = scaler_fraud.fit_transform(Xf_train_res)
Xf_test_scaled = scaler_fraud.transform(Xf_test)

scaler_credit = StandardScaler()
Xc_train_scaled = scaler_credit.fit_transform(Xc_train_res)
Xc_test_scaled = scaler_credit.transform(Xc_test)


In [ ]:
# Logistic Regression
lr_fraud = LogisticRegression(max_iter=1000, random_state=42)
lr_fraud.fit(Xf_train_scaled, yf_train_res)

lr_credit = LogisticRegression(max_iter=1000, random_state=42)
lr_credit.fit(Xc_train_scaled, yc_train_res)


In [ ]:
#  Random Forest (Ensemble Model)
rf_fraud = RandomForestClassifier(n_estimators=100, random_state=42)
rf_fraud.fit(Xf_train_res, yf_train_res)

rf_credit = RandomForestClassifier(n_estimators=100, random_state=42)
rf_credit.fit(Xc_train_res, yc_train_res)


In [ ]:
#  Evaluate models - Fraud Data
evaluate_model(lr_fraud, Xf_test_scaled, yf_test, "Logistic Regression - Fraud")
evaluate_model(rf_fraud, Xf_test, yf_test, "Random Forest - Fraud")


In [ ]:
#  Save best models and scalers
joblib.dump(rf_fraud, "../../models/rf_fraud_model.pkl")
joblib.dump(scaler_fraud, "../../models/scaler_fraud.pkl")

joblib.dump(rf_credit, "../../models/rf_credit_model.pkl")
joblib.dump(scaler_credit, "../../models/scaler_credit.pkl")
